# MUJOCO Environments

This notebook demonstrates the use of MUJOCO environments from Gymnasium. Notice that these environments have a more complex initialization process as they require a 3D graphics accelerator to run.

Check that you have a GPU available before running this notebook!

## Install dependencies

This notebook tests the InvertedPendulum environment. This environment is based on the Mujoco library. Mujoco is a library for the physics simulation of multibody systems such as robots. It contains a simulation and a visualization engine.
The visualization engine works with OpenGL, a hardware accelerated library to render 3D graphics.

Unfortunately, OpenGL does not play very nice with the Google Colab environment, as this is defined as a headless system (a system without a screen), and OpenGL usually renders its ouputs to a screen.

For this reason the installation is more complex than in other cases.

In [1]:
# First we install mujoco and gymnasium including the mujoco environments
!pip install mujoco
!pip install gymnasium[mujoco]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 18.9 MB/s eta 0:00:00


## Check if installation was successful

In this case, we are going to test that the installation is correct and that we are running using a GPU.

In [2]:
from google.colab import files

import distutils.util
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')
# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

Setting environment variable to use GPU rendering:
env: MUJOCO_GL=egl
Checking that the installation succeeded:
Installation successful.


## Helper functions

In [3]:
import cv2
import matplotlib.pyplot as plt

# Check if we running in Google Colab or Jupyter Notebook
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Do you need to connect with Google Drive? Uncomment the next two lines
    # from google.colab import drive
    # drive.mount('/content/drive')
    # This auxiliary function simplifies the visualization of OpenCV Images
    from google.colab.patches import cv2_imshow
else:
    print('Running in Jupyter Notebook')
    # This auxiliary function simplifies the visualization of OpenCV Images
    def cv2_imshow(img, title=''):
        if img.ndim > 2:
            img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img)
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()
        else:
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img, cmap='gray')
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()

# Helper functions to save videos and images
def save_video(img_array, path='/content/test.mp4'):
  height, width, layers = img_array[0].shape
  size = (width, height)
  out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MP4V'), 15, size)
  if out.isOpened():
    for i in range(len(img_array)):
      bgr_img = cv2.cvtColor(img_array[i], cv2.COLOR_RGB2BGR)
      out.write(bgr_img)
    out.release()
    print('Video saved.')
  else:
    print(f'Could not save video path: {path}')

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def to_html_video(img_array):
  # Function to update each frame in the animation
  def update_frame(i):
      frame.set_data(img_array[i])
      return frame,

  # Set up the plot
  # Your list of images
  # images = [img1, img2, img3, ...]
  # Assuming all images have the same shape as the first image
  img_height, img_width = img_array[0].shape[:2]

  # Set up the figure size based on the image size with a desired DPI
  dpi = 100
  fig_width = img_width / float(dpi)
  fig_height = img_height / float(dpi)
  fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)
  ax.axis('off')

  frame = ax.imshow(img_array[0])

  # Create an animation
  ani = FuncAnimation(fig, update_frame, frames=len(img_array), blit=True)

  # To display the animation inline, convert it to HTML and display it
  html = HTML(ani.to_html5_video())
  plt.close(fig)
  return html

Running in Google Colab



# Testing the environment

Let's test that the environment is working using random actions:

In [4]:
import gymnasium as gym
done = False

img_array=[]

env = gym.make("InvertedPendulum-v5", render_mode="rgb_array")
env.reset()

while not done:
    action = env.action_space.sample()

    obs, reward, terminated, truncated, info = env.step(action)
    frame = env.render()
    img_array.append(frame)

    done = terminated or truncated

#save_video(img_array, path='/content/random_pendulum.mp4')
to_html_video(img_array)